# Personal experiment: Row-wise centroid + PD line following

**Purpose:** test an alternative to the team's contour-based detector. This notebook estimates the line position from black pixels in several horizontal image rows, smooths the estimate over time, and uses proportional-derivative (PD) steering.

**Safety:** run cells in order, not Run All. Keep the JetBot lifted or wheels clear for the first motor-direction test. The preview cell never commands motion. Only run the driving cell after confirming the preview and wheel direction. Start with a short duration and low speeds. Stop the robot immediately if it behaves unexpectedly.

This is a standalone experiment and does not implement green-marker turns or gap crossing. Record the actual tests and outcomes before describing them in the report.

In [ ]:
import time
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()
camera = Camera.instance(width=224, height=224)

latest = {"frame": None, "timestamp": None}
def receive_frame(change):
    latest["frame"] = change["new"]
    latest["timestamp"] = time.monotonic()

camera.observe(receive_frame, names="value")
preview = widgets.Image(format="jpeg", width=336, height=336)
status = widgets.Label(value="Waiting for camera frame")
display(widgets.VBox([preview, status]))
time.sleep(0.5)
print("Frame ready:", latest["frame"] is not None)

## 1. Detector: estimate the line from pixels, not contours

For each selected image row, threshold dark pixels and calculate their horizontal centroid. Reject rows with too few dark pixels so shadows/noise are less likely to dominate. Combine the valid row estimates, giving more weight to lower rows because they are closer to the robot.

In [ ]:
# Tune while stationary. This threshold is an initial value, not a calibration.
BLACK_THRESHOLD = 75
ROW_FRACTIONS = [0.58, 0.68, 0.78, 0.86]  # image rows, from farther to nearer
MIN_BLACK_PIXELS = 4


def detect_line_centroid(frame):
    """Return (normalized_error, overlay, mask, row_results).
    Error is -1..+1: negative = line left of image center; positive = right.
    """
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    row_results = []
    weighted_sum = 0.0
    weight_total = 0.0

    for idx, frac in enumerate(ROW_FRACTIONS):
        y = int(frac * h)
        # A thin horizontal strip makes the estimate less sensitive to one pixel row.
        y0, y1 = max(0, y - 2), min(h, y + 3)
        strip = mask[y0:y1, :]
        xs = np.where(strip > 0)[1]
        cx = None
        if xs.size >= MIN_BLACK_PIXELS:
            cx = float(np.mean(xs))
            normalized = (cx - (w - 1) / 2.0) / (w / 2.0)
            weight = idx + 1  # nearer rows get higher weight
            weighted_sum += weight * normalized
            weight_total += weight
            cv2.circle(overlay, (int(cx), y), 4, (0, 255, 0), -1)
        else:
            cv2.circle(overlay, (w // 2, y), 3, (0, 0, 255), -1)
        cv2.line(overlay, (0, y), (w - 1, y), (255, 0, 0), 1)
        row_results.append((frac, cx, int(xs.size)))

    cv2.line(overlay, (w // 2, 0), (w // 2, h - 1), (0, 255, 255), 1)
    error = None if weight_total == 0 else weighted_sum / weight_total
    return error, overlay, mask, row_results


def show_detector(seconds=8):
    """Stationary-only preview. No motor commands are issued."""
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        frame = latest["frame"]
        stamp = latest["timestamp"]
        if frame is None or stamp is None or time.monotonic() - stamp > 0.5:
            status.value = "No fresh camera frame"
            time.sleep(0.05)
            continue
        error, overlay, mask, rows = detect_line_centroid(frame)
        preview.value = bgr8_to_jpeg(overlay)
        if error is None:
            status.value = "Line not found in selected rows"
        else:
            status.value = "Error: {:+.3f} | rows (y, centroid, black pixels): {}".format(
                error, [(round(y, 2), None if x is None else round(x, 1), n) for y, x, n in rows])
        time.sleep(0.04)
    robot.stop()
    print("Preview ended; motors remain stopped.")

show_detector(seconds=8)

## 2. PD controller with smoothed error

The proportional term responds to current line offset. The derivative term responds to how quickly the offset is changing, which may reduce overshoot. A short exponential moving average suppresses frame-to-frame noise. These gains are starting points and must be tuned on the real robot.

In [ ]:
BASE_SPEED = 0.10
MAX_SPEED = 0.18
KP = 0.16
KD = 0.025
SMOOTHING = 0.35       # new-sample fraction; lower = smoother but slower
LOST_LINE_TIMEOUT = 0.30
FRAME_TIMEOUT = 0.50


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def pd_commands(error, previous_error, dt):
    derivative = 0.0 if previous_error is None or dt <= 0 else (error - previous_error) / dt
    steering = KP * error + KD * derivative
    # With the common JetBot convention, positive error means line is right.
    # This sign must be verified with a wheels-off-ground test before driving.
    left = clamp(BASE_SPEED + steering, -MAX_SPEED, MAX_SPEED)
    right = clamp(BASE_SPEED - steering, -MAX_SPEED, MAX_SPEED)
    return left, right, steering

print("Controller defined. No motors commanded by this cell.")

## 3. Very short driving test (explicitly start this cell)

First confirm that the detected centroid follows the physical tape in the preview. Raise the wheels and test that a line placed to the right produces the correct steering direction. If steering is reversed, stop and swap the `+ steering` / `- steering` signs in `pd_commands`. Then place the robot on a simple, high-contrast straight line and use a 2-second test. Keep a hand near the power switch. Missing line, stale camera, or exceptions stop the motors.

In [ ]:
def run_pd_line_follow(seconds=2.0):
    robot.stop()
    start = time.monotonic()
    deadline = start + seconds
    previous_time = None
    previous_error = None
    smooth_error = None
    lost_since = None
    frame_stamp_seen = None
    try:
        while time.monotonic() < deadline:
            now = time.monotonic()
            frame = latest["frame"]
            stamp = latest["timestamp"]
            if frame is None or stamp is None or now - stamp > FRAME_TIMEOUT:
                robot.stop()
                status.value = "STOP: stale/missing camera frame"
                break
            if stamp == frame_stamp_seen:
                time.sleep(0.01)
                continue
            frame_stamp_seen = stamp
            error, overlay, mask, rows = detect_line_centroid(frame)
            preview.value = bgr8_to_jpeg(overlay)
            if error is None:
                if lost_since is None:
                    lost_since = now
                robot.stop()
                status.value = "Line not detected; stopped"
                if now - lost_since >= LOST_LINE_TIMEOUT:
                    break
                continue
            lost_since = None
            smooth_error = error if smooth_error is None else (
                SMOOTHING * error + (1.0 - SMOOTHING) * smooth_error)
            dt = 0.04 if previous_time is None else max(0.001, now - previous_time)
            left, right, steering = pd_commands(smooth_error, previous_error, dt)
            robot.set_motors(left, right)
            status.value = "e={:+.3f}, steer={:+.3f}, L={:+.2f}, R={:+.2f}".format(
                smooth_error, steering, left, right)
            previous_error = smooth_error
            previous_time = now
            time.sleep(0.01)
    except KeyboardInterrupt:
        print("Interrupted by user.")
    except Exception as exc:
        print("Safety stop after error:", repr(exc))
    finally:
        robot.stop()
        print("Motors stopped.")

# Intentionally commented out. Uncomment only after stationary preview and steering-direction check:
# run_pd_line_follow(seconds=2.0)

## 4. Cleanup

Run when finished or before switching notebooks. This stops motors and releases this notebook's camera observer and camera stream.

In [ ]:
robot.stop()
try:
    camera.unobserve(receive_frame, names="value")
except Exception:
    pass
try:
    camera.stop()
except Exception as exc:
    print("Camera stop note:", repr(exc))
print("Cleanup complete. If another notebook is using the camera, shut down this kernel first.")